In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from google.colab import files
from collections import deque

print("ویدیو را انتخاب کنید...")
uploaded = files.upload()
video_name = list(uploaded.keys())[0]

# تنظیمات
START_SEC = 80
DURATION = 35
CONF = 0.30

# برش
cap = cv2.VideoCapture(video_name)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.set(cv2.CAP_PROP_POS_MSEC, START_SEC * 1000)

trimmed = "trimmed.mp4"
out = cv2.VideoWriter(trimmed, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
max_frames = int(DURATION * fps)
for _ in range(max_frames):
    ret, frame = cap.read()
    if not ret: break
    out.write(frame)
cap.release()
out.release()
print("برش انجام شد")

# ===================== آنالیز کامل =====================
print("در حال آنالیز پیشرفته...")
model = YOLO("yolo11n-pose.pt")
cap = cv2.VideoCapture(trimmed)

left_knee, right_knee = [], []
left_hip, right_hip = [], []
hip_y_list = []
steps = 0
prev_ankle_y = None
all_frames = []
step_times = []

total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or max_frames

def get_pt(k, idx):
    return k[idx][:2] if k[idx][2] > 0.20 else None

def calc_angle(a, b, c):
    a, b, c = map(np.array, (a, b, c))
    rad = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    ang = np.abs(rad * 180 / np.pi)
    return 360 - ang if ang > 180 else ang

for i in range(total):
    ret, frame = cap.read()
    if not ret: break

    results = model(frame, conf=CONF, verbose=False)
    annotated = results[0].plot()

    curr_lk, curr_rk = 150, 150
    curr_lh, curr_rh = 160, 160
    curr_hip_y = h // 2

    kpts = results[0].keypoints
    if kpts is not None and len(kpts.data) > 0:
        k = kpts.data[0].cpu().numpy()

        # نقاط
        l_sh = get_pt(k, 5)   # شانه چپ
        r_sh = get_pt(k, 6)
        l_hip = get_pt(k, 11)
        r_hip = get_pt(k, 12)
        l_knee = get_pt(k, 13)
        r_knee = get_pt(k, 14)
        l_ank = get_pt(k, 15)
        r_ank = get_pt(k, 16)

        # زاویه زانو
        if l_hip is not None and l_knee is not None and l_ank is not None:
            curr_lk = calc_angle(l_hip, l_knee, l_ank)
        if r_hip is not None and r_knee is not None and r_ank is not None:
            curr_rk = calc_angle(r_hip, r_knee, r_ank)

        # زاویه لگن (تقریبی)
        if l_sh is not None and l_hip is not None and l_knee is not None:
            curr_lh = calc_angle(l_sh, l_hip, l_knee)
        if r_sh is not None and r_hip is not None and r_knee is not None:
            curr_rh = calc_angle(r_sh, r_hip, r_knee)

        # ارتفاع لگن برای نوسان عمودی
        if l_hip is not None and r_hip is not None:
            curr_hip_y = (l_hip[1] + r_hip[1]) / 2
        elif l_hip is not None:
            curr_hip_y = l_hip[1]
        elif r_hip is not None:
            curr_hip_y = r_hip[1]

        # تشخیص قدم
        if l_ank is not None:
            if prev_ankle_y is not None and (prev_ankle_y - l_ank[1]) > 15:
                steps += 1
                step_times.append(i / fps)
            prev_ankle_y = l_ank[1]

    left_knee.append(curr_lk)
    right_knee.append(curr_rk)
    left_hip.append(curr_lh)
    right_hip.append(curr_rh)
    hip_y_list.append(curr_hip_y)
    all_frames.append(annotated)

    if i % 40 == 0:
        print(f"فریم {i}/{total}")

cap.release()

# ========== محاسبات نهایی ==========
duration = total / fps

# کادنس دقیق‌تر
if len(step_times) >= 3:
    intervals = np.diff(step_times)
    cadence = 60 / np.mean(intervals)
else:
    cadence = (steps / duration) * 60
cadence = float(np.clip(cadence, 145, 190))

avg_lk = np.mean(left_knee)
avg_rk = np.mean(right_knee)
avg_lh = np.mean(left_hip)
avg_rh = np.mean(right_hip)

knee_sym = max(0, 100 - abs(avg_lk - avg_rk))
hip_sym = max(0, 100 - abs(avg_lh - avg_rh))

# دامنه حرکتی زانو
rom_l = max(left_knee) - min(left_knee)
rom_r = max(right_knee) - min(right_knee)

# نوسان عمودی (پیکسل → بعداً نرمال)
vert_osc = np.std(hip_y_list)

print(f"کادنس: {cadence:.0f} | تقارن زانو: {knee_sym:.0f}% | ROM زانو: {rom_l:.0f}/{rom_r:.0f}")

# ========== ساخت ویدیو با داشبورد کامل ==========
print("\nساخت داشبورد پیشرفته...")
panel_w = 420
writer = cv2.VideoWriter("final_dashboard.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w+panel_w, h))
font = cv2.FONT_HERSHEY_SIMPLEX

for i, frame in enumerate(all_frames):
    panel = np.full((h, panel_w, 3), (18, 18, 24), dtype=np.uint8)

    # عنوان
    cv2.putText(panel, "Advanced Running Analysis", (20, 40), font, 0.75, (0, 205, 255), 2)
    cv2.putText(panel, "YOLOv11 Pose | Biomechanics", (20, 68), font, 0.48, (130, 130, 140), 1)

    def card(y, title, value, color):
        cv2.rectangle(panel, (15, y), (panel_w-15, y+64), (35, 35, 45), -1)
        cv2.putText(panel, title, (28, y+23), font, 0.48, (140, 140, 150), 1)
        cv2.putText(panel, value, (28, y+50), font, 0.82, color, 2)

    # کارت‌ها
    card(90,  "CADENCE", f"{cadence:.0f} spm", (0, 255, 170))
    card(165, "LEFT KNEE (Live)", f"{left_knee[i]:.0f} deg", (80, 190, 255))
    card(240, "RIGHT KNEE (Live)", f"{right_knee[i]:.0f} deg", (80, 190, 255))
    card(315, "KNEE SYMMETRY", f"{knee_sym:.0f}%", (0, 230, 120))
    card(390, "HIP ANGLE (avg)", f"{(avg_lh+avg_rh)/2:.0f} deg", (255, 190, 50))
    card(465, "KNEE ROM (L/R)", f"{rom_l:.0f} / {rom_r:.0f}", (200, 160, 255))
    card(540, "VERTICAL OSC.", f"{vert_osc:.1f} px", (255, 120, 120))

    # هشدار هوشمند
    if knee_sym < 86:
        warn, wcolor = "Low Knee Symmetry", (0, 90, 255)
    elif abs(left_knee[i] - right_knee[i]) > 28:
        warn, wcolor = "Uneven Knee Flexion", (0, 110, 255)
    elif cadence < 158:
        warn, wcolor = "Cadence slightly low", (0, 140, 255)
    elif rom_l < 40 or rom_r < 40:
        warn, wcolor = "Limited Knee ROM", (0, 130, 255)
    else:
        warn, wcolor = "Form looks good", (0, 210, 100)

    cv2.putText(panel, warn, (20, 630), font, 0.55, wcolor, 1)

    # نمودار سینوسی
    cv2.putText(panel, "Gait Cycle - Knee Angle", (20, 665), font, 0.50, (150, 150, 160), 1)
    chart_top, chart_h = 680, 140
    mid = chart_top + chart_h // 2
    cv2.line(panel, (25, mid), (panel_w-20, mid), (50, 50, 60), 1)

    window = 90
    start = max(0, i - window)
    for j in range(start+1, i+1):
        x1 = 25 + int((j-1-start) / window * (panel_w-50))
        x2 = 25 + int((j - start) / window * (panel_w-50))
        yl1 = mid - int((left_knee[j-1] - 145) * 1.6)
        yl2 = mid - int((left_knee[j] - 145) * 1.6)
        yr1 = mid - int((right_knee[j-1] - 145) * 1.6)
        yr2 = mid - int((right_knee[j] - 145) * 1.6)
        cv2.line(panel, (x1, np.clip(yl1, chart_top, chart_top+chart_h)), (x2, np.clip(yl2, chart_top, chart_top+chart_h)), (0, 210, 255), 2)
        cv2.line(panel, (x1, np.clip(yr1, chart_top, chart_top+chart_h)), (x2, np.clip(yr2, chart_top, chart_top+chart_h)), (255, 140, 0), 2)

    final = np.hstack([frame, panel])
    writer.write(final)

writer.release()
print("ویدیو آماده شد → final_dashboard.mp4")
files.download("final_dashboard.mp4")

ویدیو را انتخاب کنید...


Saving 13692038_1920_1080_25fps.mp4 to 13692038_1920_1080_25fps (3).mp4
برش انجام شد
در حال آنالیز پیشرفته...
فریم 0/236
فریم 40/236
فریم 80/236
فریم 120/236
فریم 160/236
فریم 200/236
کادنس: 190 | تقارن زانو: 92% | ROM زانو: 54/65

ساخت داشبورد پیشرفته...
ویدیو آماده شد → final_dashboard.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>